# MedViLL Pre-training — Colab (T4 GPU)

**Yêu cầu:** Runtime → Change runtime type → **T4 GPU**

| Bước | Nội dung |
|------|----------|
| 1 | Kiểm tra GPU |
| 2 | Mount Google Drive (lấy **data**) |
| 3 | Clone code từ **GitHub** (lấy code mới nhất) |
| 4 | Cài dependencies |
| 5 | Symlink data từ Drive vào thư mục code |
| 6 | Kiểm tra data |
| 7 | Chạy training |
| 8 | (Tuỳ chọn) Lưu model ra Drive |

In [1]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU")

print(f"✅ GPU   : {torch.cuda.get_device_name(0)}")
print(f"✅ VRAM  : {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
print(f"✅ CUDA  : {torch.version.cuda}")
print(f"✅ PyTorch: {torch.__version__}")

RuntimeError: ❌ Không có GPU! Vào Runtime → Change runtime type → T4 GPU

## Bước 2: Mount Google Drive (lấy data)

In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# ============================================================
# ⚠️  CHỈ CẦN THAY DÒNG NÀY nếu thư mục ảnh không nằm tại MyDrive/PKD-NCKH/data
# Đây là thư mục chứa DATA trên Drive (ảnh + JSONL)
DRIVE_DATA_DIR = '/content/drive/MyDrive/PKD-NCKH/data'
# ============================================================

if not os.path.exists(DRIVE_DATA_DIR):
    print("❌ Không tìm thấy thư mục data. Các folder trong MyDrive:")
    for d in sorted(os.listdir('/content/drive/MyDrive')):
        print(f"   /content/drive/MyDrive/{d}")
    raise FileNotFoundError(f"Không tìm thấy: {DRIVE_DATA_DIR}")

print(f"✅ Drive data dir: {DRIVE_DATA_DIR}")
print(f"✅ Nội dung:")
for item in sorted(os.listdir(DRIVE_DATA_DIR)):
    print(f"   {item}")

## Bước 3: Clone code từ GitHub

**Thay `GITHUB_USERNAME` và `REPO_NAME` theo repo của bạn.**  
Mỗi lần chạy lại cell này sẽ `git pull` để lấy code mới nhất.

In [ ]:
import os

# ============================================================
# ⚠️  THAY 2 DÒNG NÀY
GITHUB_USERNAME = 'YOUR_USERNAME'   # GitHub username của bạn
REPO_NAME       = 'PKD-NCKH'        # Tên repo trên GitHub
BRANCH          = 'main'             # Tên branch (main hoặc master)
# ============================================================

WORK_DIR = f'/content/{REPO_NAME}'

if os.path.exists(WORK_DIR):
    print("Repo đã tồn tại → pull code mới nhất...")
    os.chdir(WORK_DIR)
    ret = os.system(f'git pull origin {BRANCH}')
    if ret != 0:
        print("⚠️  Pull thất bại, thử reset hard...")
        os.system(f'git fetch origin {BRANCH}')
        os.system(f'git reset --hard origin/{BRANCH}')
else:
    print("Cloning repo từ GitHub...")
    ret = os.system(f'git clone -b {BRANCH} https://github.com/{GITHUB_USERNAME}/{REPO_NAME}.git {WORK_DIR}')
    if ret != 0:
        raise RuntimeError(f"❌ Clone thất bại! Kiểm tra GITHUB_USERNAME='{GITHUB_USERNAME}' và REPO_NAME='{REPO_NAME}'")
    os.chdir(WORK_DIR)

os.chdir(WORK_DIR)
print(f"\n✅ Đang ở: {os.getcwd()}")
print(f"✅ Commit mới nhất:")
os.system('git log --oneline -3')

## Bước 4: Cài dependencies

In [ ]:
%%bash
pip install -q \
    transformers==4.40.0 \
    fuzzywuzzy \
    python-Levenshtein \
    pyyaml \
    tqdm \
    Pillow \
    pandas \
    scikit-learn

echo "✅ Dependencies installed"

## Bước 5: Symlink data từ Drive vào thư mục code

Thay vì copy (tốn dung lượng + thời gian), tạo symlink để code đọc thẳng từ Drive.

In [ ]:
import os, shutil

# DRIVE_DATA_DIR đã set ở Bước 2
# WORK_DIR đã set ở Bước 3
WORK_DIR       = f'/content/{REPO_NAME}'
LOCAL_DATA_DIR = os.path.join(WORK_DIR, 'data')

# Xoá thư mục data rỗng trong repo (git clone có thể tạo ra)
if os.path.exists(LOCAL_DATA_DIR) and not os.path.islink(LOCAL_DATA_DIR):
    shutil.rmtree(LOCAL_DATA_DIR)
    print(f"Đã xóa thư mục data rỗng trong repo")

# Tạo symlink: WORK_DIR/data → DRIVE_DATA_DIR
if not os.path.islink(LOCAL_DATA_DIR):
    os.symlink(DRIVE_DATA_DIR, LOCAL_DATA_DIR)
    print(f"✅ Symlink tạo thành công:")
    print(f"   {LOCAL_DATA_DIR}")
    print(f"   → {DRIVE_DATA_DIR}")
else:
    print(f"✅ Symlink đã tồn tại: {LOCAL_DATA_DIR} → {os.readlink(LOCAL_DATA_DIR)}")

# Xác nhận
print(f"\n✅ Nội dung data/:")
for item in sorted(os.listdir(LOCAL_DATA_DIR)):
    print(f"   {item}")

## Bước 6: Kiểm tra data + config

In [ ]:
import json, os, yaml, torch

WORK_DIR = f'/content/{REPO_NAME}'
os.chdir(WORK_DIR)

# --- Kiểm tra JSONL (dùng data/dataset/openi/ - dataset lớn hơn, ảnh 100% khớp) ---
print("=== JSONL files ===")
for split in ['Train', 'Valid', 'Test']:
    p = f'data/dataset/openi/{split}.jsonl'
    if os.path.exists(p):
        with open(p) as f:
            n = sum(1 for _ in f)
        print(f"  ✅ {p}: {n} samples")
    else:
        print(f"  ❌ Không thấy: {p}")

# --- Kiểm tra ảnh ---
print("\n=== Ảnh ===")
img_dir = 'data/dataset/open_i/512_3ch'
if os.path.exists(img_dir):
    imgs = [f for f in os.listdir(img_dir) if f.endswith(('.jpg', '.png'))]
    print(f"  ✅ {img_dir}: {len(imgs)} ảnh")
else:
    print(f"  ❌ Không thấy thư mục: {img_dir}")

# --- Kiểm tra 1 sample khớp ảnh ---
print("\n=== Sample check ===")
with open('data/dataset/openi/Train.jsonl') as f:
    s = json.loads(f.readline())
img_file = os.path.basename(s.get('img', ''))
full     = os.path.join(img_dir, img_file)
print(f"  img field  : {s.get('img')}")
print(f"  text[:80]  : {s.get('text','')[:80]}")
print(f"  file tìm   : {full}")
print(f"  tồn tại?   : {'✅ Có' if os.path.exists(full) else '❌ Không'}")

# --- Config + tự điều chỉnh batch_size theo VRAM và seq_len=128 ---
print("\n=== Config ===")
cfg_path = 'configs/pretrain.yaml'
with open(cfg_path) as f:
    cfg = yaml.safe_load(f)

vram_gb   = torch.cuda.get_device_properties(0).total_memory / 1024**3
seq_len   = cfg.get('seq_len', 128)

# seq_len=128 cần VRAM nhiều hơn seq_len=64 → giảm batch_size
# Effective batch luôn = 128 nhờ gradient_accumulation_steps=2
if vram_gb >= 14:
    best_bs, best_ga = 64, 2    # effective = 128
elif vram_gb >= 10:
    best_bs, best_ga = 32, 4    # effective = 128
elif vram_gb >= 6:
    best_bs, best_ga = 16, 8    # effective = 128
else:
    best_bs, best_ga = 8,  16   # effective = 128

print(f"  VRAM       : {vram_gb:.1f} GB")
print(f"  seq_len    : {seq_len}")
print(f"  batch_size : {best_bs} × grad_accum {best_ga} = effective {best_bs*best_ga}")

changed = False
if cfg.get('batch_size') != best_bs:
    print(f"  ⚠️  batch_size: {cfg['batch_size']} → {best_bs}")
    cfg['batch_size'] = best_bs
    changed = True
if cfg.get('gradient_accumulation_steps') != best_ga:
    print(f"  ⚠️  grad_accum: {cfg.get('gradient_accumulation_steps')} → {best_ga}")
    cfg['gradient_accumulation_steps'] = best_ga
    changed = True

if changed:
    with open(cfg_path, 'w') as f:
        yaml.dump(cfg, f, default_flow_style=False, allow_unicode=True)
    print("  ✅ Config đã cập nhật")
else:
    print(f"  ✅ Config OK")


## Bước 7: Chạy Training

Model checkpoint lưu vào Drive theo từng epoch → **không mất khi Colab reset**.

In [ ]:
import os

WORK_DIR = f'/content/{REPO_NAME}'
os.chdir(WORK_DIR)

# Output lưu thẳng vào Drive để không mất khi reset
OUTPUT_PATH = DRIVE_DATA_DIR.replace('/data', '/output/run1')
os.makedirs(OUTPUT_PATH, exist_ok=True)

print(f"Working dir : {WORK_DIR}")
print(f"Output sẽ lưu tại: {OUTPUT_PATH}")
print("=" * 60)
print("Bắt đầu training...")
print("=" * 60)

os.system(
    f"python {WORK_DIR}/main.py "
    f"--config {WORK_DIR}/configs/pretrain.yaml "
    f"--output_path {OUTPUT_PATH}"
)

## (Tuỳ chọn) Theo dõi VRAM — chạy bất cứ lúc nào

In [ ]:
import torch
total    = torch.cuda.get_device_properties(0).total_memory / 1024**3
alloc    = torch.cuda.memory_allocated()     / 1024**3
reserved = torch.cuda.memory_reserved()      / 1024**3
peak     = torch.cuda.max_memory_allocated() / 1024**3
print(f"Total   : {total:.2f} GB")
print(f"Alloc   : {alloc:.2f} GB")
print(f"Reserved: {reserved:.2f} GB")
print(f"Peak    : {peak:.2f} GB")
print(f"Free    : {total - reserved:.2f} GB")